# 9_旋转位置编码：长文本的长久之路

在之前的章节中，我们已经见识了 APE（绝对位置编码）的便捷。但是，面对现代大模型动辄 128k 甚至百万级的上下文，传统的绝对标签已经无法承受。本章我们将揭示 **RoPE (Rotary Positional Embedding)** 的几何美学。

### 🌀 几何直觉：位置即相位
与其给每个 Token 贴上“第一号”、“第二号”这种死的标签，RoPE 赋予了每个 Token 一个旋转的角度。
- **APE (Absolute)**: 像是在电影院的座位上贴编号。一旦座位满了（比如 1024），你就没法给第 1025 个人安排位置。
- **RoPE (Rotary)**: 像是在时钟的刻度上。Token 根据位置 $n$ 旋转特定的角度 $	heta$。Query 和 Key 之间的“相对位置”，本质上就是它们旋转后的**角度之差**。

这种“旋转不变性”使得模型具备了天然的长文本外推（Extrapolation）能力。

![image.png](images/rope_01.png)

## 📊 1. 前置准备

进行环境准备，以及模型载入：

1. 载入基于APE的微调后模型：IDEA-CCNL/Yuyuan-GPT2-110M-SciFi-Chinese（输入长度1024字符）
2. 载入基于RoPE的微调后模型：Qwen/Qwen2-0.5B-Instruct

In [ ]:
# 🔇 静默警告：我们将优先使用镜像站，并屏蔽 Hugging Face 的 Token 认证与频率限制警告。
import os
import torch
import warnings
import logging
from pathlib import Path
from transformers import AutoTokenizer, AutoModelForCausalLM
from huggingface_hub import snapshot_download
import matplotlib.pyplot as plt
import numpy as np

logging.getLogger("huggingface_hub").setLevel(logging.ERROR)
warnings.filterwarnings("ignore", message="You are sending unauthenticated requests")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"✅ 环境已就绪。运行设备: {device}")

# 🚀 方案：双效极致加载器 (下载 + 持久化 + 加载)
# 选项: "huggingface" | "hf_mirror" (推荐)
DOWNLOAD_SOURCE = "hf_mirror" 

def load_model_with_persistence(repo_id):
    model_name = repo_id.split("/")[-1]
    local_path = Path("models") / model_name
    
    # 完整性校验：必须有 config 和权重
    has_config = (local_path / "config.json").exists()
    has_weights = any(local_path.glob("*.bin")) or any(local_path.glob("*.safetensors"))
    
    if not (has_config and has_weights):
        print(f"🚀 正在补全模型权重 {model_name}...")
        local_path.mkdir(parents=True, exist_ok=True)
        if DOWNLOAD_SOURCE == "hf_mirror":
            os.environ["HF_ENDPOINT"] = "https://hf-mirror.com"
        
        snapshot_download(repo_id, local_dir=str(local_path), 
                          local_dir_use_symlinks=False, ignore_patterns=["*.msgpack", "*.h5"])
    else:
        print(f"✅ 模型就绪: {local_path}")
        
    tokenizer = AutoTokenizer.from_pretrained(str(local_path))
    model = AutoModelForCausalLM.from_pretrained(str(local_path)).to(device)
    return model, tokenizer

# 执行预加载
print("正在载入马斯克帝国科幻模型...")
model_ape, tokenizer_ape = load_model_with_persistence("IDEA-CCNL/Yuyuan-GPT2-110M-SciFi-Chinese")
model_rope, tokenizer_rope = load_model_with_persistence("Qwen/Qwen2-0.5B-Instruct")

## 2. 📚 语料与评估逻辑深度解析

### 🌌 测试语料
为了公平且硬核地测试位置编码，我们需要一段具有以下特征的语料：
1. **长程逻辑依赖**：科幻小说（如这篇关于马斯克月球工厂与戴森球布局的作品）包含了大量的技术术语（如“拉格朗日点”、“相位旋转”）和跨越数千字的因果链条，并且是弱者（IDEA-CCNL/Yuyuan-GPT2-110M-SciFi-Chinese）的强项。
2. **场景复杂性**：Gemini 3 Flash 续写的这段文字具有极高的文本密度。如果模型无法有效利用长文本窗口，它会在预测后半段的术语时产生极大的 Loss。
3. **压力测试点**：这篇 4500 字的语料确保了即使是最节省 Token 的 Qwen2，也必须在超过 1000 个位置的深度运行，从而强制触发 APE (Yuyuan) 的物理边界。
4. **内容设置**：这篇语料前半部分为主要内容（测试语义理解），后半较长的篇幅为不断重复地内容（评估长文一致性）。

### 📊 模型能力评估函数：`calculate_losses`
该函数不仅是一个计算工具，它是在模拟人类阅读时的“预测直觉”：
- **核心指标：外推损耗 (Perplexity Loss)**。我们不测试生成的“美感”，而是测试生成的“准确性”。
- **Shift 错位比对**：通过将输出与真实标签进行 1 个单位的错位切片，我们能精确量化模型在第 $n$ 个位置对第 $n+1$ 个词的‘惊讶程度’。
- **物理感知**：函数内含自动探测机制，能识别并尊重 APE 模型的物理上限（1024），确保实验数据的科学性与严谨性。

In [ ]:
# 同步读取语料
with open("data/musk_scifi_long_context.txt", "r", encoding="utf-8") as f:
    long_text = f.read()
print(f"📖 语料就绪，共计 {len(long_text)} 字符。")

def calculate_losses(model, tokenizer, text, max_len=1500):
    model.eval()
    
    # 获取物理上限 (APE 受限，RoPE 无限)
    limit = getattr(model.config, "max_position_embeddings", 1024)
    if limit is None: limit = 1024
    
    actual_len = min(max_len, limit) if "qwen" not in model.name_or_path.lower() else max_len
    if actual_len < max_len:
        print(f"⚠️ APE 物理上限已触达: {limit}")

    inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=actual_len).to(device)
    input_ids = inputs["input_ids"]
    
    with torch.no_grad():
        outputs = model(input_ids, labels=input_ids)
        probs = torch.nn.functional.cross_entropy(outputs.logits[..., :-1, :].view(-1, outputs.logits.size(-1)), 
                                                  input_ids[..., 1:].view(-1), reduction="none")
    return probs.float().cpu().numpy()

print("🏁 启动长文本 PPL 擂台赛...")
losses_ape = calculate_losses(model_ape, tokenizer_ape, long_text)
losses_rope = calculate_losses(model_rope, tokenizer_rope, long_text)
print("✅ 计算完成。")


## 3. 实验设计解析：为什么我们要计算逐位 Loss？

在本次对比实验中，我们并没有让模型直接“生成”故事，而是计算了它在阅读马斯克科幻语料时的**预测损失（Loss）**。这背后的设计逻辑如下：

### 🔄 偏移预测逻辑 (Shifted Prediction)
LLM 的核心是“根据前 $n$ 个词预测第 $n+1$ 个词”。
- 我们的代码中对 `logits` 和 `labels` 进行了 `Shift` 处理：
  - `logits` (预测输出) 移位掉最后一个词。
  - `labels` (真实标签) 移位掉第一个词。
- 这样，第 1 个位置的预测结果会与第 2 个位置的真实词进行比对。这种错位比对是验证自回归模型性能的标准手段。

### 📉 Loss 代表了什么？
- **Loss 越低**：说明模型对下一个词的预测越“笃定”，说明它理解了上下文的逻辑。
- **Loss 突增**：说明模型感到了“极大的意外（Surprise）”。

### 🎨 为什么要看 Position-Loss 曲线？
在绝对位置编码（APE）模型中，一旦 Token 的位置超过了训练时的窗口边界（例如 1024）：
1. 模型由于从未见过这个位置的编码，它输出的特征向量会变得“毫无物理意义”。
2. 这会导致预测结果瞬间崩溃，**Loss 曲线突然没了**。
3. 而旋转位置编码（RoPE）通过角度旋转来表达相对距离，它天生支持在未见过的长度上进行逻辑推演。这就是为什么 Qwen2 的折线在 1024 之后依然能保持优雅的平稳。

### 💡 专家视角：深度理解困惑度 (Perplexity)

刚才我们观察到了 Loss 曲线的波动，但它背后的物理意义是什么？

**困惑度（PPL）** 本质上是模型在预测下一个 Token 时的“有效选择数量”：
1. **确定性与分布**：
   - **低 PPL**：模型对下一个 Token 非常确定。概率分布呈现“尖峰”状计算，Top-k 概率极高。
   - **高 PPL**：模型感到非常困惑。概率分布变得极为“平坦”，模型觉得自己像是在几十个平衡的选项中进行随机抽奖。
2. **数学直觉**：
   - 公式：$PPL = e^{Loss}$。这意味着 Loss 的线性增长，反映在模型主观感受上是指数级的混乱。
3. **为什么 PPL 会在长文本末尾爆炸？**
   - 在 APE 模型（如 Yuyuan）中，由于位置编码在 1024 之后变成了完全陌生的信号，模型失去了语义锚点。它对下一个词的预测变成了盲目的随机猜测，概率分布彻底铺平，从而导致 PPL 呈指数级飙升。

![image.png](images/ppl_01.png)

In [ ]:
plt.figure(figsize=(12, 6))
plt.plot(losses_ape, label="Yuyuan-GPT2 (APE)", color="red", alpha=0.6)
plt.plot(losses_rope, label="Qwen2 (RoPE)", color="blue", alpha=0.6)
plt.axvline(x=1024, color='gray', linestyle='--', label="Yuyuan-GPT2 Physical Limit")

plt.title("Position-wise Loss (PPL): APE vs RoPE", fontsize=14)
plt.xlabel("Token Position", fontsize=12)
plt.ylabel("Loss", fontsize=12)
plt.legend()
plt.grid(True, linestyle="--", alpha=0.6)
plt.show()

print("结果清晰可见：")
print("APE 模型在 1022-1024 位置遭遇‘时空断崖’，而采用旋转编码的 Qwen2 则平滑地延续了理解。")


## 4. 🎥 终局解析：为什么 Qwen2 的曲线会“归零”？

在实验结果的最右侧，你可能会惊讶地发现 **Qwen2 的 Loss 曲线几乎贴在了横轴上（趋近于 0）**。这背后隐藏着现代 LLM 最核心的“超能力”：

### 语料的“完美复刻”测试
为了进行极端压力测试，我们手动将科幻语料进行了三倍拼接。这意味着从第 1500 个字符开始，内容其实是第 1 个字符的**完美重复**。

### 旋转位置编码的“精准定位”
- **Qwen2 (RoPE)**：由于旋转位置编码提供了极佳的“相对位置感知”，Qwen2 能够跨越数千个 Token，精准地捕捉到当前的叙事节奏与 1000 位之前的内容是完全一致的。
- **注意力机制的胜利**：当模型发现规律后，它会利用注意力机制（Attention）直接从历史缓存中提取答案。对于这种 100% 重复的模式，Qwen2 的预测准确度会无限接近 100%，从而导致 **Loss 跌至近乎 0**。

### APE 的“盲区”对比
再看 **Yuyuan-GPT2 (APE)**：尽管它也是一个优秀的模型，但受限于绝对位置编码，它在 1024 之后就失去了“坐标感”。
- 对它而言，后半段的重复语料并不是“老朋友”，而是一段完全陌生的、没有坐标支撑的信息。
- 因此，尽管内容在重复，它也无法利用 1000 位之前的历史来加速预测。它的 Loss 依然在高位挣扎，直到最终逻辑崩溃。

### 🛡️ 结论：旋转跳跃不停歇
这不仅仅是 Loss 的减少，这是**长程记忆能力**的胜利。旋转位置编码让模型拥有了“无论走多远都能回头看清原点”的能力，这也是为什么现代大模型能够写出逻辑严密的万字长文，而早期 GPT 模型只能在几百字内徘徊的根本原因。